# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0203

In [1]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupShuffleSplit, GroupKFold
from scipy.stats import pearsonr, spearmanr


In [2]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated02\community_index_v9.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target — mean: {df['community_suppression'].mean():.3f}  "
      f"std: {df['community_suppression'].std():.3f}  "
      f"range: [{df['community_suppression'].min():.3f}, {df['community_suppression'].max():.3f}]")

Dataset shape: 48,000 rows × 168 columns
Target — mean: 0.670  std: 0.377  range: [0.000, 0.999]


In [3]:
# feature matrix

r0_cols = [f"R0_c{k}" for k in range(1, 21)]

active_cols = [f"active_c{k}" for k in range(1, 21)]

diversity_col = ['diversity_level']


feature_cols = r0_cols + active_cols + diversity_col

print(f"Feature count:          {len(feature_cols)}  (vs 42 in nutrient+size, 142 in full model)")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  diversity_level:      {len(diversity_col)}")
print(f"  community_size:       0  ← removed (biological, not chemical)")
print(f"  Species membership:   0  ← removed")

# Diagnostic: confirm all feature columns exist in the loaded file
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print(f"\nWARNING: {len(missing)} missing columns: {missing}")
else:
    print("\nAll feature columns confirmed present in file.")

X = df[feature_cols]
y = df['community_suppression']

print(f"\nX shape: {X.shape}    y shape: {y.shape}")

Feature count:          41  (vs 42 in nutrient+size, 142 in full model)
  R0 concentrations:    20
  Active flags:         20
  diversity_level:      1
  community_size:       0  ← removed (biological, not chemical)
  Species membership:   0  ← removed

All feature columns confirmed present in file.

X shape: (48000, 41)    y shape: (48000,)


In [4]:
strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)


groups = df['broth_id'].values # used later for groupkfold

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"Unique broths: {df['broth_id'].nunique()}")
print(f"\nDiversity level distribution in test set:")
print(df.loc[X_test.index, 'diversity_level'].value_counts().sort_index())

Train set: 38,400 rows
Test set:  9,600 rows
Unique broths: 32

Diversity level distribution in test set:
diversity_level
1     3000
2     2400
4     1800
8     1200
16    1200
Name: count, dtype: int64


In [5]:
# baseline models (lasso and elastic net added instead of just ridge)

from sklearn.linear_model import RidgeCV

print("Baseline comparisons")

# Baseline 1: Global mean predictor 
y_mean      = y_train.mean()
y_pred_mean = np.full_like(y_test, fill_value=y_mean, dtype=float)
r2_mean     = r2_score(y_test, y_pred_mean)
print(f"Baseline (global mean predictor):  R² = {r2_mean:.4f}")


scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Baseline 2: Ridge 
ridge_alphas = np.logspace(-3, 3, 100)

ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_cv.predict(X_test_scaled)
r2_ridge = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"Ridge (alpha={ridge_cv.alpha_:.5f}):         R² = {r2_ridge:.4f}   MAE = {mae_ridge:.4f}")

# Baseline 3: Lasso with cross-validated alpha selection.
# LassoCV tries 50 alpha values on a log scale and picks the one with
# the lowest 5-fold CV error. max_iter=10000 prevents convergence warnings
# with 142 features.
lasso_cv       = LassoCV(cv=5, max_iter=10000, n_alphas=50, random_state=42, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)
y_pred_lasso   = lasso_cv.predict(X_test_scaled)
r2_lasso       = r2_score(y_test, y_pred_lasso)
mae_lasso      = mean_absolute_error(y_test, y_pred_lasso)
n_lasso_kept   = int(np.sum(lasso_cv.coef_ != 0))
print(f"Lasso (alpha={lasso_cv.alpha_:.5f}):       R² = {r2_lasso:.4f}   MAE = {mae_lasso:.4f}   features kept = {n_lasso_kept}/{len(feature_cols)}")

# Baseline 4: Elastic Net with cross-validated alpha and l1_ratio selection.
# l1_ratio controls the L1/L2 mix: 1.0 = pure Lasso, 0.0 = pure Ridge.
enet_cv        = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95],
                               cv=5, max_iter=10000, n_alphas=50,
                               random_state=42, n_jobs=-1)
enet_cv.fit(X_train_scaled, y_train)
y_pred_enet    = enet_cv.predict(X_test_scaled)
r2_enet        = r2_score(y_test, y_pred_enet)
mae_enet       = mean_absolute_error(y_test, y_pred_enet)
n_enet_kept    = int(np.sum(enet_cv.coef_ != 0))
print(f"ElasticNet (alpha={enet_cv.alpha_:.5f}, l1={enet_cv.l1_ratio_:.2f}): R² = {r2_enet:.4f}   MAE = {mae_enet:.4f}   features kept = {n_enet_kept}/{len(feature_cols)}")

Baseline comparisons
Baseline (global mean predictor):  R² = -0.0002
Ridge (alpha=30.53856):         R² = 0.8107   MAE = 0.0992
Lasso (alpha=0.00019):       R² = 0.8108   MAE = 0.0997   features kept = 29/41
ElasticNet (alpha=0.00039, l1=0.50): R² = 0.8108   MAE = 0.0997   features kept = 30/41


In [6]:
# Which species did Lasso keep?
lasso_coef      = pd.Series(lasso_cv.coef_, index=feature_cols)
lasso_nonzero   = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)

r0_kept         = int((lasso_coef[r0_cols] != 0).sum())
print(f"\nLasso feature selection: R0 kept={r0_kept}/20   Active kept={int((lasso_coef[active_cols] != 0).sum())}/20")
print("Lasso top 10 features by |coefficient|:")
for feat, coef in lasso_nonzero.head(10).items():
    ftype = ("R0" if feat in r0_cols
             
             else "Active" if feat in active_cols else "Structural")
    print(f"  {feat:<22}  coef={coef:>8.5f}  [{ftype}]")


Lasso feature selection: R0 kept=15/20   Active kept=14/20
Lasso top 10 features by |coefficient|:
  R0_c11                  coef=-0.17513  [R0]
  R0_c17                  coef=-0.13761  [R0]
  R0_c1                   coef=-0.13750  [R0]
  R0_c7                   coef= 0.11918  [R0]
  R0_c4                   coef=-0.10071  [R0]
  active_c9               coef= 0.10018  [Active]
  active_c15              coef=-0.08853  [Active]
  active_c11              coef= 0.08555  [Active]
  active_c5               coef= 0.08161  [Active]
  active_c4               coef=-0.07069  [Active]


In [10]:
# xgboost initial

# With only 41 features and a harder prediction task (no size signal),
# we allow deeper trees (max_depth=8) so the model can still capture
# complex nutrient interaction patterns.
model_init = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_init = model_init.predict(X_test)
r2_init = r2_score(y_test, y_pred_init)
mae_init = mean_absolute_error(y_test, y_pred_init)
mse_init = mean_squared_error(y_test, y_pred_init)

print(f"XGBoost (initial):  R² = {r2_init:.4f}   MAE = {mae_init:.4f}   MSE = {mse_init:.5f}")
print(f"Best iteration: {model_init.best_iteration}")

XGBoost (initial):  R² = 0.8107   MAE = 0.0995   MSE = 0.02713
Best iteration: 131


In [8]:
# hyperparameter tuning

param_grid = {                     # using winning hyperparams from first run tuning
    'max_depth':        [5],

    'learning_rate':    [0.1],
    'subsample':        [0.8],
    'colsample_bytree': [0.75],
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = xgb.XGBRegressor(
    n_estimators=200,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits

Best parameters:
  colsample_bytree: 0.75
  learning_rate: 0.1
  max_depth: 5
  subsample: 0.8
Best CV RMSE: 0.1640


In [11]:
best_params = grid_search.best_params_

model_final = xgb.XGBRegressor(
    **best_params,
    n_estimators=1000,
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=40,
    verbosity=0
)

# Pass TWO eval sets: training first, test second.
# This records both training loss and validation loss per round,
# which we need to draw the training curve.
# validation_0 = train loss, validation_1 = test (validation) loss.
model_final.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

y_pred_final = model_final.predict(X_test)
r2_final     = r2_score(y_test, y_pred_final)
mae_final    = mean_absolute_error(y_test, y_pred_final)
mse_final    = mean_squared_error(y_test, y_pred_final)
rmse_final   = np.sqrt(mse_final)

print(f"\nFinal Model Performance")
print(f"R²:   {r2_final:.4f}")
print(f"MAE:  {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"MSE:  {mse_final:.5f}")
print(f"Best iteration (trees used): {model_final.best_iteration}")

print("\nModel Comparison")
print(f"{'Model':<30} {'R²':>8}  {'MAE':>8}")
print("-" * 48)
print(f"{'Global mean (baseline)':<30} {r2_mean:>8.4f}  {'N/A':>8}")
print(f"{'Ridge Regression':<30} {r2_ridge:>8.4f}  {mae_ridge:>8.4f}")
print(f"{'Lasso':<30} {r2_lasso:>8.4f}  {mae_lasso:>8.4f}")
print(f"{'Elastic Net':<30} {r2_enet:>8.4f}  {mae_enet:>8.4f}")
print(f"{'XGBoost (initial)':<30} {r2_init:>8.4f}  {mae_init:>8.4f}")
print(f"{'XGBoost (tuned)':<30} {r2_final:>8.4f}  {mae_final:>8.4f}")

# training curve
evals    = model_final.evals_result_
tr_rmse  = evals['validation_0']['rmse']   # training loss per round
val_rmse = evals['validation_1']['rmse']   # validation loss per round
rounds   = range(1, len(tr_rmse) + 1)
best_it  = model_final.best_iteration

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(rounds, tr_rmse,  color='steelblue',  lw=1.5, label='Train RMSE')
ax_tc.plot(rounds, val_rmse, color='darkorange', lw=1.5, label='Validation RMSE')
ax_tc.axvline(best_it + 1, color='red', linestyle='--', lw=1.2,
              label=f'Best iteration = {best_it + 1}')
ax_tc.set_xlabel('Boosting round')
ax_tc.set_ylabel('RMSE')
ax_tc.set_title(f'XGBoost Training Curve  --  best iter={best_it + 1},  val RMSE={val_rmse[best_it]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('xgb_training_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: xgb_training_curve.png")

# groupkfold

gkf          = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []

print("\nGroupKFold evaluation (held-out broths):")
for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    held_broths = sorted(set(df.iloc[te_idx]['broth_id']))
    held_divs   = sorted(df.iloc[te_idx]['diversity_level'].unique())
    print(f"\nFold {fold_num}: broths={held_broths}  diversity levels={held_divs}  n_test={len(X_te):,}")

    m_fold = xgb.XGBRegressor(
        **best_params,
        n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
        random_state=42, n_jobs=-1,
        eval_metric='rmse', early_stopping_rounds=40, verbosity=0
    )
    m_fold.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

    yp_fold  = m_fold.predict(X_te)
    r2_f     = r2_score(y_te, yp_fold)
    mae_f    = mean_absolute_error(y_te, yp_fold)
    rmse_f   = np.sqrt(mean_squared_error(y_te, yp_fold))

    fold_r2.append(r2_f); fold_mae.append(mae_f); fold_rmse.append(rmse_f)
    print(f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  RMSE={rmse_f:.4f}")

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:   mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:  mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE: mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(f"Drop: {r2_final - np.mean(fold_r2):.4f}  (how much was broth memorisation)")

[0]	validation_0-rmse:0.34851	validation_1-rmse:0.35055
[50]	validation_0-rmse:0.16451	validation_1-rmse:0.16527
[100]	validation_0-rmse:0.16379	validation_1-rmse:0.16470
[126]	validation_0-rmse:0.16379	validation_1-rmse:0.16471

Final Model Performance
R²:   0.8107
MAE:  0.1000
RMSE: 0.1647
MSE:  0.02712
Best iteration (trees used): 86

Model Comparison
Model                                R²       MAE
------------------------------------------------
Global mean (baseline)          -0.0002       N/A
Ridge Regression                 0.8107    0.0992
Lasso                            0.8108    0.0997
Elastic Net                      0.8108    0.0997
XGBoost (initial)                0.8107    0.0995
XGBoost (tuned)                  0.8107    0.1000
Saved: xgb_training_curve.png

GroupKFold evaluation (held-out broths):

Fold 1: broths=['Broth_02', 'Broth_07', 'Broth_12', 'Broth_17', 'Broth_22', 'Broth_27', 'Broth_32']  diversity levels=[np.int64(1), np.int64(2), np.int64(4), np.int64(8), 

In [12]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.8717  0.0842  0.1606   0.9337    0.8211   0.0049  0.4483  3,000   
D=2     0.8446  0.0884  0.1625   0.9190    0.9307   0.0004  0.4123  2,400   
D=4     0.1137  0.1016  0.1654   0.3439    0.5204   0.0092  0.1757  1,800   
D=8     0.3413  0.1182  0.1633   0.5859    0.6658  -0.0069  0.2012  1,200   
D=16    0.0045  0.1425  0.1788   0.0761    0.0636   0.0018  0.1792  1,200   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.5625  0.1499  0.2463   0.8128    0.8017   0.1097  0.3724  1,920
N=16    0.7499  0.1093  0.1843   0.8776    0.8396   0.0509  0.3684  1,920
N=24 

In [13]:
# shap analysis

print("SHAP analysis")

explainer = shap.TreeExplainer(model_final)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nAll 41 features ranked by mean |SHAP|:")
print(shap_importance.round(5).to_string())

r0_shap     = shap_importance[r0_cols].sum()
active_shap = shap_importance[active_cols].sum()
div_shap    = shap_importance[diversity_col].sum()
total_shap  = r0_shap + active_shap + div_shap

print(f"\nSHAP by group:")
print(f"  R0 concentrations:  {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:       {active_shap:.4f}  ({100*active_shap/total_shap:.1f}%)")
print(f"  diversity_level:    {div_shap:.4f}  ({100*div_shap/total_shap:.1f}%)")

SHAP analysis

All 41 features ranked by mean |SHAP|:
R0_c4              0.09534
R0_c11             0.05367
R0_c1              0.04941
R0_c17             0.04537
diversity_level    0.01661
R0_c3              0.01629
R0_c8              0.01440
R0_c16             0.01245
R0_c7              0.01102
R0_c2              0.00985
R0_c10             0.00980
R0_c15             0.00882
R0_c9              0.00720
R0_c19             0.00655
active_c1          0.00640
R0_c12             0.00530
R0_c13             0.00485
active_c11         0.00449
active_c19         0.00446
R0_c14             0.00372
active_c12         0.00334
active_c4          0.00317
active_c14         0.00303
active_c17         0.00224
active_c15         0.00201
R0_c5              0.00166
active_c5          0.00028
active_c16         0.00027
active_c13         0.00024
active_c8          0.00022
R0_c6              0.00018
active_c7          0.00015
R0_c20             0.00015
active_c9          0.00014
R0_c18             0.00010
a

In [15]:
# figures

from matplotlib.patches import Patch

print("\n" + "=" * 60)
print("SECTION 11: Generating plots → pure_nutrient_results_updated02.png")
print("=" * 60)

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'XGBoost PURE NUTRIENT (Chemistry-Only) Suppression Predictor — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
    f'[nutrient+size R²≈?  |  full model R²=0.930]',
    fontsize=12, fontweight='bold', y=0.99
)

# Panel 1: Predicted vs Actual coloured by diversity level
ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = df.loc[X_test.index, 'diversity_level'].values == d
    ax1.scatter(y_test.values[mask], y_pred_final[mask],
                alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted suppression')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

# Panel 2: Residuals coloured by community size
# Even though size wasn't a feature, colouring by it reveals whether
# the model is systematically biased against certain community sizes.
ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - y_pred_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = df.loc[X_test.index, 'community_size'].values == n
    ax2.scatter(y_pred_final[mask], resid[mask],
                alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residuals (coloured by size — not a feature)')
ax2.legend(fontsize=7, markerscale=3)

# Panel 3: SHAP importance — all 41 features
ax3 = fig.add_subplot(2, 3, 3)
def feat_colour(f):
    if f in r0_cols:     return '#4CAF50'
    if f in active_cols: return '#FF9800'
    return '#9C27B0'   # diversity_level
top_n   = len(feature_cols)
top_shap = shap_importance.head(top_n)
cols = [feat_colour(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=6)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('All 41 Features by SHAP')
ax3.legend(handles=[
    Patch(fc='#4CAF50', label='R0 concentration'),
    Patch(fc='#FF9800', label='Active flag'),
    Patch(fc='#9C27B0', label='diversity_level'),
], loc='lower right', fontsize=8)

# Panel 4: R² by diversity level
ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# Panel 5: R² by community size
# Key diagnostic: if R² varies strongly by size, the model is missing
# the community_size signal that we deliberately excluded.
ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N (NOT a feature in this model)')
ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size\n(size was excluded — shows what we lost)')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# Panel 6: SHAP group pie
ax6 = fig.add_subplot(2, 3, 6)
ax6.pie(
    [r0_shap, active_shap, div_shap],
    labels=['R0\nconcentrations', 'Active\nflags', 'diversity_level'],
    colors=['#4CAF50', '#FF9800', '#9C27B0'],
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax6.set_title('SHAP Importance by Feature Group\n(pure nutrient model)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('pure_nutrient_results_updated02.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: pure_nutrient_results_updated02.png")


SECTION 11: Generating plots → pure_nutrient_results_updated02.png
Saved: pure_nutrient_results_updated02.png
